In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift')

In [ ]:
!pip install xplique timm opencv-python

In [ ]:
# F Hinder et. al. Fish Head Dataset Loader and Embedder

import numpy as np

class FishHeadLoader:
    def __init__(self, path, ratio_non_drifting=0.333):
        self.df = np.load(path)
        self.q = ratio_non_drifting
    def ratio(self, ratio_non_drifting=None):
         if ratio_non_drifting is not None:
             self.q = ratio_non_drifting
    def take(self, size, create_testset=False):
        embedding,z = self.df["X"], self.df["z"]-1
        q = 1-self.q
        sel = np.hstack( [np.random.choice(np.where(z==i)[0], size=int( size *(q/2 if i != 0 else 1-q)), replace=True) for i in np.unique(z)] ).flatten()
        train_observation_state = np.zeros(sel.shape[0],dtype=int)
        train_observation_state[z[sel]==1] = 1
        train_observation_state[np.random.choice(np.where(z[sel]==0)[0],size=(z[sel]==0).sum()//2,replace=False)] = 1

        if create_testset:
            nsel = np.ones(embedding.shape[0], dtype=bool)
            nsel[sel] = False
            return (embedding[sel],train_observation_state,np.abs(z[sel]),z[sel]), (embedding[nsel],z[nsel])

        return embedding[sel],train_observation_state,np.abs(z[sel]),z[sel]

In [ ]:
loader = FishHeadLoader("/content/drive/MyDrive/data/fishHead/fish_head_embedding.npz", 0.6)


In [ ]:
np.random.seed(42)
X, sample_labels, drift_state, z_true = loader.take(size=1000)

print(f"Embeddings shape:  {X.shape}")
print(f"BD samples:        {(sample_labels==0).sum()}")
print(f"AD samples:        {(sample_labels==1).sum()}")
print(f"Drifting samples:  {drift_state.sum()}")
print(f"Non-drifting:      {(drift_state==0).sum()}")

In [ ]:
from sklearn.decomposition import NMF
from sklearn.model_selection import train_test_split

n_concepts = 10

# ReLU
X_relu = np.maximum(X, 0)

X_train, X_test, y_train, y_test = train_test_split(
    X_relu, sample_labels, train_size=0.7, random_state=42)

bd_embeddings = X_relu[sample_labels == 0]
ad_embeddings = X_relu[sample_labels == 1]

# BD
bd_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
bd_crops_u = bd_reducer.fit_transform(bd_embeddings)
bd_w = bd_reducer.components_.astype(np.float32)

# AD
ad_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
ad_crops_u = ad_reducer.fit_transform(ad_embeddings)
ad_w = ad_reducer.components_.astype(np.float32)

drift_basis = np.vstack([bd_w, ad_w])


In [ ]:
from experiment_helpers.helper_function import estimate_importance_l
from experiment_helpers.helper_function import estimate_importance_helper_l
from experiment_helpers.helper_function import l_compute_predictions
from experiment_helpers.helper_function import concept_counter
from experiment_helpers.helper_function import local_imp_concepts_probability
from experiment_helpers.driftLocalizer import Localizer
from sklearn.metrics import accuracy_score

class DummyCraft:
    def __init__(self):
        self.sensitivities = {}

drift_craft = DummyCraft()

# Localizer
localizer = Localizer(min_samples_leaf=20)
localizer.fit(X_train, y_train)

localizer_bin_preds = localizer.l_predict(X_test)
print(f"h acc: {accuracy_score(localizer_bin_preds, y_test):.3f}")

In [ ]:
drift_imp = np.round(
    estimate_importance_l(localizer, drift_craft, drift_basis, X_relu), 3)

image_drift_imp = [
    estimate_importance_helper_l(
        drift_craft, localizer, drift_basis,
        emb[np.newaxis, :],
        class_of_interest=localizer_bin_preds[i])
    for i, emb in enumerate(X_test)
]

localizer_bin_train_preds = localizer.l_predict(X_train)
image_drift_imp_l_train = [
            estimate_importance_helper_l(
                drift_craft, localizer, drift_basis,
                image, class_of_interest=localizer_bin_train_preds[i])
            for i, image in enumerate(X_train)
        ]
concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

ht_acc = local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=y_test)
ht_lp  = local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=localizer_bin_preds)

print(f"h tilde acc: {ht_acc:.3f}")
print(f"h tilde LP acc (vs h): {ht_lp:.3f}")

In [ ]:
import numpy as np
from sklearn.decomposition import NMF, non_negative_factorization
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from itertools import combinations
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

from experiment_helpers.helper_function import (
    estimate_importance_l, estimate_importance_helper_l,
    l_compute_predictions, concept_counter,
    local_imp_concepts_probability, reconstruct_inputs
)
from experiment_helpers.driftLocalizer import Localizer


class DummyCraft:
    def __init__(self):
        self.sensitivities = {}


def cosine_hungarian_loss(V1, V2):
    assert V1.shape == V2.shape
    assert len(V1.shape) == 2
    cost_matrix = 1 - cosine_similarity(V1, V2)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    loss = cost_matrix[row_ind, col_ind].sum()
    return float(loss / len(V1))


def stability(v_drift_list):
    scores = []
    n = len(v_drift_list)
    for i in range(n):
        for j in range(i+1, n):
            scores.append(cosine_hungarian_loss(v_drift_list[i], v_drift_list[j]))
    return np.mean(scores), np.std(scores)


def stability_across(result_dict, value_range):
    scores = []
    for val_a, val_b in combinations(value_range, 2):
        for va in result_dict[val_a]['v_drift_list']:
            for vb in result_dict[val_b]['v_drift_list']:
                scores.append(cosine_hungarian_loss(va, vb))
    return np.mean(scores), np.std(scores)


path = "/content/drive/MyDrive/data/fishHead/fish_head_embedding.npz"
n_concepts = 10

In [ ]:
size = 2000
fixed_ratio = 0.6
runs = 25

for run in range(runs):
    loader = FishHeadLoader(path, ratio_non_drifting=fixed_ratio)
    X, sample_labels, drift_state, z_true = loader.take(size=size)

    X_relu = np.maximum(X, 0)

    X_train, X_test, y_train, y_test = train_test_split(
        X_relu, sample_labels, train_size=0.7, random_state=42)

    bd_embeddings = X_relu[sample_labels == 0]
    ad_embeddings = X_relu[sample_labels == 1]

    bd_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
    bd_crops_u = bd_reducer.fit_transform(bd_embeddings)
    bd_w = bd_reducer.components_.astype(np.float32)

    ad_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
    ad_crops_u = ad_reducer.fit_transform(ad_embeddings)
    ad_w = ad_reducer.components_.astype(np.float32)

    drift_basis = np.vstack([bd_w, ad_w])

    result_dict_size[size]["v_drift_list"].append(drift_basis.copy())

    drift_craft = DummyCraft()

    localizer = Localizer(min_samples_leaf=20)
    localizer.fit(X_train, y_train)

    localizer_bin_preds = localizer.l_predict(X_test)
    result_dict_size[size]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

    drift_imp = np.round(estimate_importance_l(localizer, drift_craft, drift_basis, X_relu), 3)

    image_drift_imp = [
        estimate_importance_helper_l(
            drift_craft, localizer, drift_basis,
            emb[np.newaxis, :], class_of_interest=localizer_bin_preds[i])
        for i, emb in enumerate(X_test)
    ]

    localizer_bin_train_preds = localizer.l_predict(X_train)
    image_drift_imp_train = [
        estimate_importance_helper_l(
            drift_craft, localizer, drift_basis,
            emb[np.newaxis, :], class_of_interest=localizer_bin_train_preds[i])
        for i, emb in enumerate(X_train)
    ]
    concept_dist = concept_counter(image_drift_imp_train, localizer_bin_train_preds)

    result_dict_size[size]['one_local_l'].append(
        local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=y_test))
    result_dict_size[size]['one_local_l_lp'].append(
        local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=localizer_bin_preds))

    recon_single = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=1)
    result_dict_size[size]['recon_single'].append(
        accuracy_score(localizer.l_predict(recon_single), y_test))
    result_dict_size[size]['recon_single_lp'].append(
        accuracy_score(localizer.l_predict(recon_single), localizer_bin_preds))

    recon_all = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=2*n_concepts)
    result_dict_size[size]['recon_all'].append(
        accuracy_score(localizer.l_predict(recon_all), y_test))
    result_dict_size[size]['recon_all_lp'].append(
        accuracy_score(localizer.l_predict(recon_all), localizer_bin_preds))

    print(f"size = {size} , run {len(result_dict_size[size]['drift_localizer'])}/25")


In [ ]:
size_range = [200, 500, 1000, 1500, 2000]
runs = 25
fixed_ratio = 0.6

result_dict_size = {size: {
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for size in size_range}

for size in size_range:
    for run in range(runs):
        loader = FishHeadLoader(path, ratio_non_drifting=fixed_ratio)
        X, sample_labels, drift_state, z_true = loader.take(size=size)

        X_relu = np.maximum(X, 0)

        X_train, X_test, y_train, y_test = train_test_split(
            X_relu, sample_labels, train_size=0.7, random_state=42)

        bd_embeddings = X_relu[sample_labels == 0]
        ad_embeddings = X_relu[sample_labels == 1]

        bd_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
        bd_crops_u = bd_reducer.fit_transform(bd_embeddings)
        bd_w = bd_reducer.components_.astype(np.float32)

        ad_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
        ad_crops_u = ad_reducer.fit_transform(ad_embeddings)
        ad_w = ad_reducer.components_.astype(np.float32)

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_size[size]["v_drift_list"].append(drift_basis.copy())

        drift_craft = DummyCraft()

        localizer = Localizer(min_samples_leaf=20)
        localizer.fit(X_train, y_train)

        localizer_bin_preds = localizer.l_predict(X_test)
        result_dict_size[size]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer, drift_craft, drift_basis, X_relu), 3)

        image_drift_imp = [
            estimate_importance_helper_l(
                drift_craft, localizer, drift_basis,
                emb[np.newaxis, :], class_of_interest=localizer_bin_preds[i])
            for i, emb in enumerate(X_test)
        ]

        localizer_bin_train_preds = localizer.l_predict(X_train)
        image_drift_imp_train = [
            estimate_importance_helper_l(
                drift_craft, localizer, drift_basis,
                emb[np.newaxis, :], class_of_interest=localizer_bin_train_preds[i])
            for i, emb in enumerate(X_train)
        ]
        concept_dist = concept_counter(image_drift_imp_train, localizer_bin_train_preds)

        result_dict_size[size]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=y_test))
        result_dict_size[size]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=1)
        result_dict_size[size]['recon_single'].append(
            accuracy_score(localizer.l_predict(recon_single), y_test))
        result_dict_size[size]['recon_single_lp'].append(
            accuracy_score(localizer.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=2*n_concepts)
        result_dict_size[size]['recon_all'].append(
            accuracy_score(localizer.l_predict(recon_all), y_test))
        result_dict_size[size]['recon_all_lp'].append(
            accuracy_score(localizer.l_predict(recon_all), localizer_bin_preds))

        print(f"size = {size} , run {run+1}")

In [ ]:
for n in size_range:
    h_mean    = np.mean(result_dict_size[n]['drift_localizer'])
    h_std     = np.std(result_dict_size[n]['drift_localizer'])
    ht_mean   = np.mean(result_dict_size[n]['one_local_l'])
    ht_std    = np.std(result_dict_size[n]['one_local_l'])
    ht_lp_mean   = np.mean(result_dict_size[n]['one_local_l_lp'])
    ht_lp_std    = np.std(result_dict_size[n]['one_local_l_lp'])
    rs_mean   = np.mean(result_dict_size[n]['recon_single'])
    rs_std    = np.std(result_dict_size[n]['recon_single'])
    rs_lp_mean   = np.mean(result_dict_size[n]['recon_single_lp'])
    rs_lp_std    = np.std(result_dict_size[n]['recon_single_lp'])
    ra_mean   = np.mean(result_dict_size[n]['recon_all'])
    ra_std    = np.std(result_dict_size[n]['recon_all'])
    ra_lp_mean   = np.mean(result_dict_size[n]['recon_all_lp'])
    ra_lp_std    = np.std(result_dict_size[n]['recon_all_lp'])

    loss_mean, loss_std = stability(result_dict_size[n]['v_drift_list'])

    print(f"size: {n} "
          f"h = {h_mean:.3f}   {h_std:.3f}  "
          f"ht = {ht_mean:.3f}   {ht_std:.3f}  "
          f"ht_lp = {ht_lp_mean:.3f}   {ht_lp_std:.3f}  "
          f"rs = {rs_mean:.3f}   {rs_std:.3f}  "
          f"rs_lp = {rs_lp_mean:.3f}   {rs_lp_std:.3f}  "
          f"ra = {ra_mean:.3f}   {ra_std:.3f}  "
          f"ra_lp = {ra_lp_mean:.3f}   {ra_lp_std:.3f}  "
          f"stability = {loss_mean:.4f}   {loss_std:.4f}  ")

In [ ]:
loss_mean, loss_std = stability_across(result_dict_size, size_range)
print(f"Cross-size stability: {loss_mean:.4f}   {loss_std:.4f}")

In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_fishhead_size.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['size', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for size in size_range:
        for method_name in method_names:
            row = [size, method_name] + result_dict_size[size][method_name]
            writer.writerow(row)

with open('/content/drive/MyDrive/results/stability_fishhead_size.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['size', 'stability_mean', 'stability_std'])
    for size in size_range:
        loss_mean, loss_std = stability(result_dict_size[size]['v_drift_list'])
        writer.writerow([size, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_size, size_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])

In [ ]:
ratio_range = [0.2, 0.4, 0.6, 0.8]
runs = 25
fixed_size = 1000

result_dict_ratio = {ratio: {
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for ratio in ratio_range}

for ratio in ratio_range:
    for run in range(runs):
        loader = FishHeadLoader(path, ratio_non_drifting=ratio)
        X, sample_labels, drift_state, z_true = loader.take(size=fixed_size)

        X_relu = np.maximum(X, 0)

        X_train, X_test, y_train, y_test = train_test_split(
            X_relu, sample_labels, train_size=0.7, random_state=42)

        bd_embeddings = X_relu[sample_labels == 0]
        ad_embeddings = X_relu[sample_labels == 1]

        bd_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
        bd_crops_u = bd_reducer.fit_transform(bd_embeddings)
        bd_w = bd_reducer.components_.astype(np.float32)

        ad_reducer = NMF(n_components=n_concepts, alpha_W=1e-2, max_iter=2000)
        ad_crops_u = ad_reducer.fit_transform(ad_embeddings)
        ad_w = ad_reducer.components_.astype(np.float32)

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_ratio[ratio]["v_drift_list"].append(drift_basis.copy())

        drift_craft = DummyCraft()

        localizer = Localizer(min_samples_leaf=20)
        localizer.fit(X_train, y_train)

        localizer_bin_preds = localizer.l_predict(X_test)
        result_dict_ratio[ratio]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer, drift_craft, drift_basis, X_relu), 3)

        image_drift_imp = [
            estimate_importance_helper_l(
                drift_craft, localizer, drift_basis,
                emb[np.newaxis, :], class_of_interest=localizer_bin_preds[i])
            for i, emb in enumerate(X_test)
        ]

        localizer_bin_train_preds = localizer.l_predict(X_train)
        image_drift_imp_train = [
            estimate_importance_helper_l(
                drift_craft, localizer, drift_basis,
                emb[np.newaxis, :], class_of_interest=localizer_bin_train_preds[i])
            for i, emb in enumerate(X_train)
        ]
        concept_dist = concept_counter(image_drift_imp_train, localizer_bin_train_preds)

        result_dict_ratio[ratio]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=y_test))
        result_dict_ratio[ratio]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=1)
        result_dict_ratio[ratio]['recon_single'].append(
            accuracy_score(localizer.l_predict(recon_single), y_test))
        result_dict_ratio[ratio]['recon_single_lp'].append(
            accuracy_score(localizer.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test, image_drift_imp, drift_basis, num_concepts=2*n_concepts)
        result_dict_ratio[ratio]['recon_all'].append(
            accuracy_score(localizer.l_predict(recon_all), y_test))
        result_dict_ratio[ratio]['recon_all_lp'].append(
            accuracy_score(localizer.l_predict(recon_all), localizer_bin_preds))

        print(f"ratio = {ratio} , run {run+1}")

In [ ]:
loss_mean, loss_std = stability_across(result_dict_ratio, ratio_range)
print(f"Cross-ratio stability: {loss_mean:.4f}   {loss_std:.4f}")

In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_fishhead_ratio.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['ratio', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for ratio in ratio_range:
        for method_name in method_names:
            row = [ratio, method_name] + result_dict_ratio[ratio][method_name]
            writer.writerow(row)

with open('/content/drive/MyDrive/results/stability_fishhead_ratio.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['ratio', 'stability_mean', 'stability_std'])
    for ratio in ratio_range:
        loss_mean, loss_std = stability(result_dict_ratio[ratio]['v_drift_list'])
        writer.writerow([ratio, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_ratio, ratio_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])